Referência
https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/agents/gemini_data_analytics/intro_gemini_data_analytics_http.ipynb

### Var e Import

In [25]:
import google
import requests
import json
import json as json_lib
import textwrap

In [7]:
billing_project = "llm-studies"
location = "global"
api_version = "v1beta"
dataset_id = "analytics_527405664"
base_url = "https://geminidataanalytics.googleapis.com"
data_agent_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/dataAgents"
data_agent_url

'https://geminidataanalytics.googleapis.com/v1beta/projects/llm-studies/locations/global/dataAgents'

'https://geminidataanalytics.googleapis.com/v1beta/projects/llm-studies/locations/global/dataAgents'

### Auth - revisar!

In [9]:
#google.auth.default()

#access_token = !gcloud auth application-default print-access-token
#headers = {
#    "Authorization": f"Bearer {access_token[0]}",
#    "Content-Type": "application/json",
#}

In [8]:
import google.auth
import google.auth.transport.requests

credentials, project = google.auth.default()
auth_request = google.auth.transport.requests.Request()
credentials.refresh(auth_request)

headers = {
    "Authorization": f"Bearer {credentials.token}",
    "Content-Type": "application/json"
}

### Agent

In [4]:
data_agent_id = "events_data_agent"

In [4]:
datasource_references = {
    "bq": {
        "tableReferences": [
            {
                "projectId": "llm-studies",
                "datasetId": "analytics_processed",
                "tableId": "fct_events",
            }
        ]
    }
}

In [5]:
example_queries = [
    {
        "naturalLanguageQuestion": "Conte quantas linhas existem na tabela de eventos do dia 11 de março de 2026",
        "sqlQuery": "SELECT COUNT(*) AS total_eventos FROM fct_events AS t1 WHERE t1.data_evento = '2026-03-11'",
    },
    {
        "naturalLanguageQuestion": "Quantos IDs de usuários diferentes (únicos) interagiram com o app hoje?",
        "sqlQuery": "SELECT COUNT(DISTINCT t1.id_usuario) AS usuarios_unicos FROM fct_events AS t1 WHERE t1.data_evento = '2026-03-20'",
    },
    {
        "naturalLanguageQuestion": "Quais são os 10 nomes de eventos mais frequentes registrados?",
        "sqlQuery": "SELECT t1.nome_evento, COUNT(*) AS total FROM fct_events AS t1 GROUP BY t1.nome_evento ORDER BY total DESC LIMIT 10",
    },
    {
        "naturalLanguageQuestion": "Liste as 5 URLs de páginas que tiveram mais visualizações (page_view)",
        "sqlQuery": "SELECT t1.url_pagina, COUNT(*) AS visualizacoes FROM fct_events AS t1 WHERE t1.nome_evento = 'page_view' GROUP BY t1.url_pagina ORDER BY visualizacoes DESC LIMIT 5",
    },
    {
        "naturalLanguageQuestion": "Qual é a contagem de eventos agrupada por plataforma?",
        "sqlQuery": "SELECT t1.plataforma, COUNT(*) AS total_eventos FROM fct_events AS t1 GROUP BY t1.plataforma",
    }
]

glossary_terms = [
    {
        "display_name": "Data do Evento",
        "description": "A data cronológica (ano-mês-dia) em que o evento foi registrado.",
        "labels": ["Temporal", "Dimensão", "Calendário"]
    },
    {
        "display_name": "Timestamp do Evento",
        "description": "A marca temporal exata, incluindo data, hora e microssegundos, do momento da ocorrência do evento.",
        "labels": ["Temporal", "Técnico", "Precisão"]
    },
    {
        "display_name": "Nome do Evento",
        "description": "O nome identificador da ação realizada pelo usuário (ex: 'page_view', 'session_start').",
        "labels": ["Comportamento", "Dimensão", "Evento"]
    },
    {
        "display_name": "ID do Usuário",
        "description": "Identificador único atribuído a cada usuário para rastreamento de sua jornada na plataforma.",
        "labels": ["Usuário", "Identificador", "Privacidade"]
    },
    {
        "display_name": "Plataforma",
        "description": "O meio ou dispositivo através do qual o usuário acessou a aplicação (ex: 'WEB').",
        "labels": ["Tecnologia", "Dimensão", "Dispositivo"]
    },
    {
        "display_name": "URL da Página",
        "description": "O endereço web completo da página onde o evento ocorreu.",
        "labels": ["Conteúdo", "Dimensão", "Navegação"]
    },
    {
        "display_name": "ID da Sessão",
        "description": "Um identificador numérico que agrupa um conjunto de interações de um usuário em um período de tempo.",
        "labels": ["Sessão", "Identificador", "Agrupamento"]
    }
]
use_glossary_terms = True
use_example_queries = True

In [7]:
system_instruction = "Pense como um analista de negócio e marketing que entender o comportamento do usuário da aplicação"

In [9]:
data_agent_payload = {
    "name": f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}",  # Optional
    "description": "Data agent to analyze firebase events.",
    "data_analytics_agent": {
        "published_context": {
            "datasource_references": datasource_references,
            "system_instruction": system_instruction,
            "options": {
                "analysis": {
                    # Optional - if wanting to use advanced analysis with python.
                    # Default is False.
                    "python": {"enabled": False}
                }
            },
        }
    },
}

data_agent_payload["data_analytics_agent"]["published_context"]["example_queries"] = example_queries
data_agent_payload["data_analytics_agent"]["published_context"]["glossary_terms"] = glossary_terms
params = {"data_agent_id": data_agent_id}

In [14]:
data_agent_response = requests.post(
    data_agent_url, params=params, json=data_agent_payload, headers=headers
)
print(json.dumps(data_agent_response.json(), indent=2))

{
  "name": "projects/llm-studies/locations/global/operations/operation-1774031559648-64d78e8eee5ab-b27ec497-28e29c12",
  "metadata": {
    "@type": "type.googleapis.com/google.cloud.geminidataanalytics.v1beta.OperationMetadata",
    "createTime": "2026-03-20T18:32:39.754329287Z",
    "target": "projects/llm-studies/locations/global/dataAgents/events_data_agent",
    "verb": "create",
    "requestedCancellation": false,
    "apiVersion": "v1beta"
  },
  "done": false
}


In [9]:
data_agent_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}"

data_agent_response = requests.get(data_agent_url, headers=headers)

if data_agent_response.status_code == 200:
    print("Fetched Data Agent successfully!")
    print(json.dumps(data_agent_response.json(), indent=2))
else:
    print(f"Error: {data_agent_response.status_code}")
    print(data_agent_response.text)

Fetched Data Agent successfully!
{
  "name": "projects/llm-studies/locations/global/dataAgents/events_data_agent",
  "description": "Data agent to analyze firebase events.",
  "createTime": "2026-03-20T18:32:39.660998968Z",
  "updateTime": "2026-03-20T18:32:40.446539411Z",
  "dataAnalyticsAgent": {
    "publishedContext": {
      "systemInstruction": "Pense como um analista de neg\u00f3cio e marketing que entender o comportamento do usu\u00e1rio da aplica\u00e7\u00e3o",
      "options": {
        "analysis": {
          "python": {}
        }
      },
      "exampleQueries": [
        {
          "naturalLanguageQuestion": "Conte quantas linhas existem na tabela de eventos do dia 11 de mar\u00e7o de 2026",
          "sqlQuery": "SELECT COUNT(*) AS total_eventos FROM fct_events AS t1 WHERE t1.data_evento = '2026-03-11'"
        },
        {
          "naturalLanguageQuestion": "Quantos IDs de usu\u00e1rios diferentes (\u00fanicos) interagiram com o app hoje?",
          "sqlQuery": "SEL

In [11]:
conversation_id = "conversation_2"
conversation_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/conversations"
conversation_payload = {
    "agents": [
        f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}"
    ],
    "name": f"projects/{billing_project}/locations/{location}/conversations/{conversation_id}",
}
params = {"conversation_id": conversation_id}

conversation_response = requests.post(
    conversation_url, headers=headers, params=params, json=conversation_payload
)

if conversation_response.status_code == 200:
    print("Conversation created successfully!")
    print(json.dumps(conversation_response.json(), indent=2))
else:
    print(f"Error creating Conversation: {conversation_response.status_code}")
    print(conversation_response.text)

Conversation created successfully!
{
  "name": "projects/llm-studies/locations/global/conversations/conversation_2",
  "agents": [
    "projects/llm-studies/locations/global/dataAgents/events_data_agent"
  ],
  "createTime": "2026-03-26T21:03:52.812489Z",
  "lastUsedTime": "2026-03-26T21:03:52.972125Z"
}


In [12]:
conversation_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/conversations"
conversation_response = requests.get(conversation_url, headers=headers)

# Handle the response
if conversation_response.status_code == 200:
    print("Conversation fetched successfully!")
    print(json.dumps(conversation_response.json(), indent=2))
else:
    print(f"Error while fetching conversation: {conversation_response.status_code}")
    print(conversation_response.text)

Conversation fetched successfully!
{
  "conversations": [
    {
      "name": "projects/llm-studies/locations/global/conversations/conversation_2",
      "agents": [
        "projects/llm-studies/locations/global/dataAgents/events_data_agent"
      ],
      "createTime": "2026-03-26T21:03:52.812489Z",
      "lastUsedTime": "2026-03-26T21:03:52.972125Z"
    },
    {
      "name": "projects/llm-studies/locations/global/conversations/3ed2ab92-947b-43d9-9e69-3c4d1a953a8b",
      "agents": [
        "projects/llm-studies/locations/global/dataAgents/events_data_agent"
      ],
      "createTime": "2026-03-25T14:24:38.730101Z",
      "lastUsedTime": "2026-03-25T14:24:53.435Z",
      "labels": {
        "host": "bigquery"
      }
    },
    {
      "name": "projects/llm-studies/locations/global/conversations/9dd4a5a4-4a34-4cc0-9bb9-4fba84a3f9ef",
      "agents": [
        "projects/llm-studies/locations/global/dataAgents/ga_data_agent"
      ],
      "createTime": "2026-03-25T14:21:27.019411Z"

In [13]:
# @title Streaming Chat Messages

def is_json(str):
    try:
        json_object = json_lib.loads(str)
    except ValueError:
        return False
    return True

def handle_text_response(resp):
    parts = resp["parts"]
    full_text = "".join(parts)
    if "\n" not in full_text and len(full_text) > 80:
        wrapped_text = textwrap.fill(full_text, width=80)
        print(wrapped_text)
    else:
        print(full_text)

def get_property(data, field_name, default=""):
    return data[field_name] if field_name in data else default

def display_schema(data):
    fields = data["fields"]
    df = pd.DataFrame(
        {
            "Column": map(lambda field: get_property(field, "name"), fields),
            "Type": map(lambda field: get_property(field, "type"), fields),
            "Description": map(
                lambda field: get_property(field, "description", "-"), fields
            ),
            "Mode": map(lambda field: get_property(field, "mode"), fields),
        }
    )
    display(df)

def display_section_title(text):
    display(HTML(f"<h2>{text}</h2>"))

def format_bq_table_ref(table_ref):
    return "{}.{}.{}".format(
        table_ref["projectId"], table_ref["datasetId"], table_ref["tableId"]
    )

def format_looker_table_ref(table_ref):
    return "lookmlModel: {}, explore: {}".format(
        table_ref["lookmlModel"], table_ref["explore"]
    )

def display_datasource(datasource):
    source_name = ""

    if "studioDatasourceId" in datasource:
        source_name = datasource["studioDatasourceId"]
    elif "lookerExploreReference" in datasource:
        source_name = format_looker_table_ref(datasource["lookerExploreReference"])
    else:
        source_name = format_bq_table_ref(datasource["bigqueryTableReference"])

    print(source_name)
    if "schema" in datasource:
        display_schema(datasource["schema"])

def handle_schema_response(resp):
    if "query" in resp:
        print(resp["query"]["question"])
    elif "result" in resp:
        display_section_title("Schema resolved")
        print("Data sources:")
        for datasource in resp["result"]["datasources"]:
            display_datasource(datasource)

def handle_data_response(resp):
    if "query" in resp:
        query = resp["query"]
        display_section_title("Retrieval query")
        if "name" in query:
            print("Query name: {}".format(query["name"]))
        if "question" in query:
          print("Question: {}".format(query["question"]))
        if "datasources" in query:
          print("Data sources:")
          for datasource in query["datasources"]:
              display_datasource(datasource)
    elif "generatedSql" in resp:
        display_section_title("SQL generated")
        print(resp["generatedSql"])
    elif "result" in resp:
        display_section_title("Data retrieved")

        fields = map(
            lambda field: get_property(field, "name"),
            resp["result"]["schema"]["fields"],
        )
        dict = {}

        for field in fields:
            dict[field] = [get_property(el, field) for el in resp["result"]["data"]]

        display(pd.DataFrame(dict))

def handle_chart_response(resp):
    if "query" in resp:
        print(resp["query"]["instructions"])
    elif "result" in resp:
        vegaConfig = resp["result"]["vegaConfig"]
        alt.Chart.from_json(json_lib.dumps(vegaConfig)).display()

def handle_clarification_response(resp):
    display_section_title("Clarification required")

    clarification_questions = ""
    for q in resp["questions"]:
        # Add the question text
        clarification_questions += f"{q['question']}\n"
        # Add the options as bullet points
        for option in q["options"]:
            clarification_questions += f"* {option}\n"
        clarification_questions += "\n" # Add a newline between questions
    print(clarification_questions)

def handle_error(resp):
    display_section_title("Error")
    print("Code: {}".format(resp["code"]))
    print("Message: {}".format(resp["message"]))

def get_stream(url, json):
    s = requests.Session()

    acc = ""

    with s.post(url, json=json, headers=headers, stream=True) as resp:
        for line in resp.iter_lines():
            if not line:
                continue

            decoded_line = str(line, encoding="utf-8")

            if decoded_line == "[{":
                acc = "{"
            elif decoded_line == "}]":
                acc += "}"
            elif decoded_line == ",":
                continue
            else:
                acc += decoded_line

            if not is_json(acc):
                continue

            data_json = json_lib.loads(acc)

            if "systemMessage" not in data_json:
                if "error" in data_json:
                    handle_error(data_json["error"])
                continue

            if "text" in data_json["systemMessage"]:
                handle_text_response(data_json["systemMessage"]["text"])
            elif "schema" in data_json["systemMessage"]:
                handle_schema_response(data_json["systemMessage"]["schema"])
            elif "data" in data_json["systemMessage"]:
                handle_data_response(data_json["systemMessage"]["data"])
            elif "chart" in data_json["systemMessage"]:
                handle_chart_response(data_json["systemMessage"]["chart"])
            elif "clarification" in data_json["systemMessage"]:
                handle_clarification_response(data_json["systemMessage"]["clarification"])
            else:
                colored_json = highlight(
                    acc, lexers.JsonLexer(), formatters.TerminalFormatter()
                )
                print(colored_json)
            print("\n")
            acc = ""

In [15]:
# fmt: off
question = "quantos eventos"  # @param {type:"string"}
# fmt: on

chat_url = (
    f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}:chat"
)

 # Construct the payload
chat_payload = {
    "parent": f"projects/{billing_project}/locations/global",
    "messages": [{"userMessage": {"text": question}}],
    "conversation_reference": {
        "conversation": f"projects/{billing_project}/locations/{location}/conversations/{conversation_id}",
        "data_agent_context": {
            "data_agent": f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}",
        },
    },
}
# Call get_stream method to stream the response
get_stream(chat_url, chat_payload)

NameError: name 'json_lib' is not defined

In [24]:
conversation_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/conversations"


conversation_response = requests.get(conversation_url, headers=headers)

# Handle the response
if conversation_response.status_code == 200:
    print("Conversation fetched successfully!")
    print(json.dumps(conversation_response.json(), indent=2))
else:
    print(f"Error while fetching conversation: {conversation_response.status_code}")
    print(conversation_response.text)

Conversation fetched successfully!
{
  "conversations": [
    {
      "name": "projects/llm-studies/locations/global/conversations/conversation_1",
      "agents": [
        "projects/llm-studies/locations/global/dataAgents/data_agent_1"
      ],
      "createTime": "2026-03-09T16:25:46.528536Z",
      "lastUsedTime": "2026-03-09T16:25:47.051255Z"
    }
  ]
}


In [20]:
# @title Streaming Chat Messages

def is_json(str):
    try:
        json_object = json_lib.loads(str)
    except ValueError:
        return False
    return True

def handle_text_response(resp):
    parts = resp["parts"]
    full_text = "".join(parts)
    if "\n" not in full_text and len(full_text) > 80:
        wrapped_text = textwrap.fill(full_text, width=80)
        print(wrapped_text)
    else:
        print(full_text)

def get_property(data, field_name, default=""):
    return data[field_name] if field_name in data else default

def display_schema(data):
    fields = data["fields"]
    df = pd.DataFrame(
        {
            "Column": map(lambda field: get_property(field, "name"), fields),
            "Type": map(lambda field: get_property(field, "type"), fields),
            "Description": map(
                lambda field: get_property(field, "description", "-"), fields
            ),
            "Mode": map(lambda field: get_property(field, "mode"), fields),
        }
    )
    display(df)

def display_section_title(text):
    display(HTML(f"<h2>{text}</h2>"))

def format_bq_table_ref(table_ref):
    return "{}.{}.{}".format(
        table_ref["projectId"], table_ref["datasetId"], table_ref["tableId"]
    )

def format_looker_table_ref(table_ref):
    return "lookmlModel: {}, explore: {}".format(
        table_ref["lookmlModel"], table_ref["explore"]
    )

def display_datasource(datasource):
    source_name = ""

    if "studioDatasourceId" in datasource:
        source_name = datasource["studioDatasourceId"]
    elif "lookerExploreReference" in datasource:
        source_name = format_looker_table_ref(datasource["lookerExploreReference"])
    else:
        source_name = format_bq_table_ref(datasource["bigqueryTableReference"])

    print(source_name)
    if "schema" in datasource:
        display_schema(datasource["schema"])

def handle_schema_response(resp):
    if "query" in resp:
        print(resp["query"]["question"])
    elif "result" in resp:
        display_section_title("Schema resolved")
        print("Data sources:")
        for datasource in resp["result"]["datasources"]:
            display_datasource(datasource)

def handle_data_response(resp):
    if "query" in resp:
        query = resp["query"]
        display_section_title("Retrieval query")
        if "name" in query:
            print("Query name: {}".format(query["name"]))
        if "question" in query:
          print("Question: {}".format(query["question"]))
        if "datasources" in query:
          print("Data sources:")
          for datasource in query["datasources"]:
              display_datasource(datasource)
    elif "generatedSql" in resp:
        display_section_title("SQL generated")
        print(resp["generatedSql"])
    elif "result" in resp:
        display_section_title("Data retrieved")

        fields = map(
            lambda field: get_property(field, "name"),
            resp["result"]["schema"]["fields"],
        )
        dict = {}

        for field in fields:
            dict[field] = [get_property(el, field) for el in resp["result"]["data"]]

        display(pd.DataFrame(dict))

def handle_chart_response(resp):
    if "query" in resp:
        print(resp["query"]["instructions"])
    elif "result" in resp:
        vegaConfig = resp["result"]["vegaConfig"]
        alt.Chart.from_json(json_lib.dumps(vegaConfig)).display()

def handle_clarification_response(resp):
    display_section_title("Clarification required")

    clarification_questions = ""
    for q in resp["questions"]:
        # Add the question text
        clarification_questions += f"{q['question']}\n"
        # Add the options as bullet points
        for option in q["options"]:
            clarification_questions += f"* {option}\n"
        clarification_questions += "\n" # Add a newline between questions
    print(clarification_questions)

def handle_error(resp):
    display_section_title("Error")
    print("Code: {}".format(resp["code"]))
    print("Message: {}".format(resp["message"]))

def get_stream(url, json):
    s = requests.Session()

    acc = ""

    with s.post(url, json=json, headers=headers, stream=True) as resp:
        for line in resp.iter_lines():
            if not line:
                continue

            decoded_line = str(line, encoding="utf-8")

            if decoded_line == "[{":
                acc = "{"
            elif decoded_line == "}]":
                acc += "}"
            elif decoded_line == ",":
                continue
            else:
                acc += decoded_line

            if not is_json(acc):
                continue

            data_json = json_lib.loads(acc)

            if "systemMessage" not in data_json:
                if "error" in data_json:
                    handle_error(data_json["error"])
                continue

            if "text" in data_json["systemMessage"]:
                handle_text_response(data_json["systemMessage"]["text"])
            elif "schema" in data_json["systemMessage"]:
                handle_schema_response(data_json["systemMessage"]["schema"])
            elif "data" in data_json["systemMessage"]:
                handle_data_response(data_json["systemMessage"]["data"])
            elif "chart" in data_json["systemMessage"]:
                handle_chart_response(data_json["systemMessage"]["chart"])
            elif "clarification" in data_json["systemMessage"]:
                handle_clarification_response(data_json["systemMessage"]["clarification"])
            else:
                colored_json = highlight(
                    acc, lexers.JsonLexer(), formatters.TerminalFormatter()
                )
                print(colored_json)
            print("\n")
            acc = ""

In [26]:
question = "quantos eventos"  # @param {type:"string"}
# fmt: on

chat_url = (
    f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}:chat"
)

 # Construct the payload
chat_payload = {
    "parent": f"projects/{billing_project}/locations/global",
    "messages": [{"userMessage": {"text": question}}],
    "conversation_reference": {
        "conversation": f"projects/{billing_project}/locations/{location}/conversations/{conversation_id}",
        "data_agent_context": {
            "data_agent": f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}",
        },
    },
}

# Call get_stream method to stream the response
get_stream(chat_url, chat_payload)

Retrieving or searching for contextRetrieved context for 2 tables.


Analyzing Airport CountsI'm focused on executing the SQL query to determine the
top 5 states by the number of airports. I've considered the potential for a
request related to "events", but no such data exists. I am confident that the
user is interested in the top 5 states by airport count and will use the
`state_abbreviation` for grouping the data. I'm ready to proceed with generating
the bar chart.


Charting Airport DataI have successfully retrieved the data and am now building
the bar chart. I've considered the user's "quantos eventos" query and, given the
context, will proceed with visualizing the top 5 states by airport count, as the
user intended. I've made sure to include axis labels, a title, and sorted the
bars.


Generating the AnswerI'm synthesizing the output, now. The query about the bar
graph for the top 5 states and the number of events (interpreted as airports) is
coming together. The top states by air

NameError: name 'HTML' is not defined